In [1]:
!pip install fastapi uvicorn python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Environment loaded successfully!")

Environment loaded successfully!


In [3]:
!pip install ollama




[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import ollama

def ask_llm(prompt):
    response = ollama.chat(
       model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [5]:
answer = ask_llm("What is Memora?")
print(answer)

I couldn't find any information on a well-known entity or concept called "Memora." It's possible that it's a private or obscure reference, or it may be a misspelling or variation of a different term.

However, I can suggest a few possibilities:

1. Memora could be a brand or product name that I'm not familiar with.
2. It's possible that Memora is a misspelling or variation of a different term, such as "Memorial" or "Memory."
3. Memora could be a proper noun, such as a person's name or a location.

If you could provide more context or information about Memora, I may be able to help you better.


In [6]:
memories=[]
def save_memory(memory):
    memories.append(memory)
    return "Memory saved successfully."

In [7]:
save_memory("My internship application deadline is October 30.")
print(memories)

['My internship application deadline is October 30.']


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

memory_collection = chroma_client.get_or_create_collection(
    name="memora_memories"
)

In [11]:
def search_memory(query, n_results=3, memory_type=None, exclude_type=None):

    query_embedding = embedding_model.encode(query).tolist()

    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": n_results,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if memory_type:
        query_kwargs["where"] = {
            "type": memory_type
        }

    elif exclude_type:
        query_kwargs["where"] = {
            "type": {
                "$ne": exclude_type
            }
        }

    results = memory_collection.query(**query_kwargs)

    return results

In [14]:
results = search_memory("internship deadline")
print(results)

{'ids': [[]], 'embeddings': None, 'documents': [[]], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[]], 'distances': [[]]}


In [15]:
!pip install sentence-transformers


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
!pip install chromadb


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import chromadb

chroma_client = chromadb.Client()

In [19]:
memory_collection = chroma_client.create_collection(
    name="memora_memories"
)

In [20]:
memory = "My internship application deadline is October 30."
memory_embedding = embedding_model.encode(memory).tolist()

In [21]:
memory_collection.add(
    ids=["memory_001"],
    documents=[memory],
    embeddings=[memory_embedding]
)

In [22]:
query = "When do I need to submit my internship application?"

In [23]:
query_embedding = embedding_model.encode(query).tolist()

In [24]:
results = memory_collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)

print(results)

{'ids': [['memory_001']], 'embeddings': None, 'documents': [['My internship application deadline is October 30.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.5409945845603943]]}


core semantic memory mechanism


In [25]:
def save_memory(memory_id, memory):
    embedding = embedding_model.encode(memory).tolist()

    memory_collection.add(
        ids=[memory_id],
        documents=[memory],
        embeddings=[embedding]
    )

    return "Memory saved successfully."

In [26]:
save_memory(
    "memory_002",
    "My preferred programming language for AI projects is Python."
)

'Memory saved successfully.'

Semantic Search function

In [48]:
def search_memory(query, n_results=3, memory_type=None, exclude_type=None):

    query_embedding = embedding_model.encode(query).tolist()

    query_kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": n_results,
        "include": [
            "documents",
            "metadatas",
            "distances"
        ]
    }

    if memory_type:
        query_kwargs["where"] = {
            "type": memory_type
        }

    elif exclude_type:
        query_kwargs["where"] = {
            "type": {
                "$ne": exclude_type
            }
        }

    results = memory_collection.query(**query_kwargs)

    return results

In [28]:
results = search_memory("Which programming language do I prefer for AI?")
print(results["documents"])

[['My preferred programming language for AI projects is Python.', 'My internship application deadline is October 30.']]


In [29]:
memory = "My internship application deadline is October 30."

memory_embedding = embedding_model.encode(memory).tolist()

memory_collection.add(
    ids=["memory_003"],
    documents=[memory],
    embeddings=[memory_embedding],
    metadatas=[{
        "type": "deadline",
        "source": "internship_notice",
        "created_at": "2026-10-23"
    }]
)

In [30]:
results = memory_collection.get(
    ids=["memory_003"],
    include=["documents", "metadatas"]
)

print(results)

{'ids': ['memory_003'], 'embeddings': None, 'documents': ['My internship application deadline is October 30.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'type': 'deadline', 'source': 'internship_notice', 'created_at': '2026-10-23'}]}


In [31]:
from datetime import datetime


def save_memory(memory_id, memory, memory_type, source="user"):
    embedding = embedding_model.encode(memory).tolist()

    metadata = {
        "type": memory_type,
        "source": source,
        "created_at": datetime.now().isoformat()
    }

    memory_collection.add(
        ids=[memory_id],
        documents=[memory],
        embeddings=[embedding],
        metadatas=[metadata]
    )

    return "Memory saved successfully."

In [32]:
save_memory(
    "memory_004",
    "My preferred programming language for AI projects is Python.",
    "preference"
)

'Memory saved successfully.'

In [33]:
save_memory(
    "memory_005",
    "I need to submit my AI internship application by October 30.",
    "deadline",
    "internship_notice"
)

'Memory saved successfully.'

In [34]:
results = search_memory("What internships deadlines do I have?")
print(results["documents"])
print(results["metadatas"])
print(results["distances"])

[['My internship application deadline is October 30.', 'My internship application deadline is October 30.', 'I need to submit my AI internship application by October 30.']]
[[None, {'created_at': '2026-10-23', 'source': 'internship_notice', 'type': 'deadline'}, {'type': 'deadline', 'created_at': '2026-10-25T23:50:55.078009', 'source': 'internship_notice'}]]
[[0.45845335721969604, 0.45845335721969604, 0.8934810161590576]]


In [35]:
def display_memories(results):
    for i, memory in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]

        print(f"\nMemory {i + 1}")
        print(f"Text: {memory}")

        if metadata:
            print(f"Type: {metadata.get('type', 'unknown')}")
            print(f"Source: {metadata.get('source', 'unknown')}")
            print(f"Created: {metadata.get('created_at', 'unknown')}")
        else:
            print("Type: legacy memory")
            print("Source: unknown")
            print("Created: unknown")

        print(f"Distance: {distance:.4f}")

In [36]:
results = search_memory("What internships deadlines do I have?")
display_memories(results)


Memory 1
Text: My internship application deadline is October 30.
Type: legacy memory
Source: unknown
Created: unknown
Distance: 0.4585

Memory 2
Text: My internship application deadline is October 30.
Type: deadline
Source: internship_notice
Created: 2026-10-23
Distance: 0.4585

Memory 3
Text: I need to submit my AI internship application by October 30.
Type: deadline
Source: internship_notice
Created: 2026-10-25T23:50:55.078009
Distance: 0.8935


In [37]:
!pip install pymupdf


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


PDF EXTRACTION

In [38]:
import fitz
def extract_text_from_pdf(pdf_path):
    document = fitz.open(pdf_path)
    
    pages = []

    for page in document:
        text = page.get_text()
        pages.append(text)

    document.close()

    return "\n".join(pages)

In [39]:
text = extract_text_from_pdf("test.pdf")

print(text[:3000])

UNIT-1 
INTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES 
Department of Chemistry, ANITS  
Page 1 
 
 
 Environment: Environment is derived from the French word Environner, which means to 
encircle or surround. Environment is defined as the sum total of water, air, land and the inter-
relationships that exist among them and with the human beings, other living organisms and 
materials. 
 Multi-disciplinary nature of environmental Science 
Environment is complex and has multifarious aspects. 
Environmental sciences deal with working of earth, its life support systems, its interactions, 
influences, its problems, solutions. Keeping in view the complex nature of environment knowledge 
and information from various disciplines of science, social science, law and engineering have to be 
included in environmental studies to understand it completely. 
 Life sciences like botany, zoology, microbiology, genetics and bio technology etc help in 
understanding the biotic components and their inter

CHUNKING

In [40]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [41]:
chunks = chunk_text(text)

print("Number of chunks:", len(chunks))
print (chunks[0])
print("First chunk length:", len(chunks[0]))

Number of chunks: 47
UNIT-1 
INTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES 
Department of Chemistry, ANITS  
Page 1 
 
 
 Environment: Environment is derived from the French word Environner, which means to 
encircle or surround. Environment is defined as the sum total of water, air, land and the inter-
relationships that exist among them and with the human beings, other living organisms and 
materials. 
 Multi-disciplinary nature of environmental Science 
Environment is complex and has multifarious aspects. 
Environmental sciences deal with working of earth, its life support systems, its interactions, 
influences, its problems, solutions. Keeping in view the complex nature of environment knowledge 
and information from various disciplines of science, social science, law and engineering have to be 
included
First chunk length: 800


In [42]:
print(chunks[0])
print("\n--- NEXT CHUNK ---\n")
print(chunks[1])

UNIT-1 
INTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES 
Department of Chemistry, ANITS  
Page 1 
 
 
 Environment: Environment is derived from the French word Environner, which means to 
encircle or surround. Environment is defined as the sum total of water, air, land and the inter-
relationships that exist among them and with the human beings, other living organisms and 
materials. 
 Multi-disciplinary nature of environmental Science 
Environment is complex and has multifarious aspects. 
Environmental sciences deal with working of earth, its life support systems, its interactions, 
influences, its problems, solutions. Keeping in view the complex nature of environment knowledge 
and information from various disciplines of science, social science, law and engineering have to be 
included

--- NEXT CHUNK ---

mation from various disciplines of science, social science, law and engineering have to be 
included in environmental studies to understand it completely. 
 Life sciences like b

In [43]:
def add_pdf_to_memory(chunks, pdf_name):
    ids = []
    documents = []
    embeddings = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        chunk_id = f"{pdf_name}_{i}"

        embedding = embedding_model.encode(chunk).tolist()

        ids.append(chunk_id)
        documents.append(chunk)
        embeddings.append(embedding)

        metadatas.append({
            "type": "document_chunk",
            "source": pdf_name,
            "chunk_index": i
        })

    memory_collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    return f"Added {len(chunks)} chunks from {pdf_name}."

In [44]:
result = add_pdf_to_memory(chunks, "test.pdf")
print(result)

Added 47 chunks from test.pdf.


In [45]:
results = search_memory(
    "What is discussed in this document?",
    n_results=3
)

display_memories(results)


Memory 1
Text: f localized importance. e.g. Impact of 
mining or hydro-electric project in an area, problems of disposal and management of solid waste, 
river and lake pollution, soil erosion, water logging and salinity of soil, fluorosis problem in local 
population etc. For dealing with local environmental issues we have to think and act locally. 
    It is absolutely essential to create awareness about environment because- 
 Environment belongs to all and is important to all: Whatever is the occupation or age of a 
person, he will be affected by environment and also he will affect the environment by his deeds. 
 Problems due to Wake of Development: Development, in its wake gave birth to Urbanization, 
Industrial Growth, Transportation Systems, Agriculture and Housing etc. However, it has become 
pha
Type: document_chunk
Source: test.pdf
Created: unknown
Distance: 1.4927

Memory 2
Text: Indiato give 
a mandate for creating environmental among all citizens of India. 
 Sh. Rajender

till here it was just pdf into text into chunking ----> embedding -----> chromadb -------> retrieval relevant chunks 
now User Question
      ↓
Semantic Retrieval
      ↓
Relevant PDF chunks
      ↓
Llama 3.1
      ↓
Natural-language answer

In [49]:
def rag_answer(question, n_results=3):
    results = search_memory(
    question,
    n_results=n_results,
    memory_type="document_chunk"
)

    retrieved_chunks = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks):

        metadata = retrieved_metadata[i]

        source = metadata.get("source", "Unknown")
        page = metadata.get("page", "Unknown")

        context_parts.append(
            f"[Source: {source}, Page: {page}]\n{chunk}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
You are Memora, a personal knowledge assistant.

Answer the user's question using ONLY the information
provided in the context.

If the answer cannot be found in the context, say:
"I couldn't find that information in your stored knowledge."

Context:
{context}

User question:
{question}

Answer clearly and concisely.
"""

    answer = ask_llm(prompt)

    return {
        "answer": answer,
        "sources": retrieved_metadata
    }

In [50]:
answer = rag_answer("What is environment?")
print(answer)

{'answer': 'Environment is defined as the sum total of water, air, land and the inter-relationships that exist among them and with human beings, other living organisms, and materials.', 'sources': [{'chunk_index': 0, 'source': 'test.pdf', 'type': 'document_chunk'}, {'source': 'test.pdf', 'type': 'document_chunk', 'chunk_index': 2}, {'source': 'test.pdf', 'chunk_index': 7, 'type': 'document_chunk'}]}


In [51]:
import fitz


def extract_pdf_pages(pdf_path):
    document = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(document, start=1):
        text = page.get_text()

        if text.strip():
            pages.append({
                "page": page_number,
                "text": text.strip()
            })

    document.close()

    return pages

In [52]:
pages = extract_pdf_pages("test.pdf")

print("Number of pages:", len(pages))
print(pages[0]["page"])
print(pages[0]["text"][:500])

Number of pages: 12
1
UNIT-1 
INTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES 
Department of Chemistry, ANITS  
Page 1 
 
 
 Environment: Environment is derived from the French word Environner, which means to 
encircle or surround. Environment is defined as the sum total of water, air, land and the inter-
relationships that exist among them and with the human beings, other living organisms and 
materials. 
 Multi-disciplinary nature of environmental Science 
Environment is complex and has multifarious aspects. 
Env


In [53]:
def chunk_pdf_pages(pages, chunk_size=800, overlap=100):
    chunks = []

    for page in pages:
        page_text = page["text"]
        page_number = page["page"]

        start = 0

        while start < len(page_text):
            end = start + chunk_size
            chunk_text_content = page_text[start:end].strip()

            if chunk_text_content:
                chunks.append({
                    "text": chunk_text_content,
                    "page": page_number
                })

            start += chunk_size - overlap

    return chunks

In [54]:
pdf_chunks = chunk_pdf_pages(pages)

print("Number of chunks:", len(pdf_chunks))

Number of chunks: 52


In [55]:
print(pdf_chunks[0])

{'text': 'UNIT-1 \nINTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES \nDepartment of Chemistry, ANITS  \nPage 1 \n \n \n\uf076 Environment: Environment is derived from the French word Environner, which means to \nencircle or surround. Environment is defined as the sum total of water, air, land and the inter-\nrelationships that exist among them and with the human beings, other living organisms and \nmaterials. \n\uf076 Multi-disciplinary nature of environmental Science \nEnvironment is complex and has multifarious aspects. \nEnvironmental sciences deal with working of earth, its life support systems, its interactions, \ninfluences, its problems, solutions. Keeping in view the complex nature of environment knowledge \nand information from various disciplines of science, social science, law and engineering have to be \nincluded', 'page': 1}


store these chunks in chromadb

In [56]:
def add_pdf_chunks_to_memory(pdf_chunks, pdf_name):
    ids = []
    documents = []
    embeddings = []
    metadatas = []

    for i, chunk in enumerate(pdf_chunks):

        chunk_id = f"{pdf_name}_{i}"

        embedding = embedding_model.encode(
            chunk["text"]
        ).tolist()

        ids.append(chunk_id)
        documents.append(chunk["text"])
        embeddings.append(embedding)

        metadatas.append({
            "type": "document_chunk",
            "source": pdf_name,
            "page": chunk["page"],
            "chunk_index": i
        })

    memory_collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )

    return f"Added {len(pdf_chunks)} chunks from {pdf_name}."

In [57]:
chroma_client.delete_collection("memora_memories")

memory_collection = chroma_client.create_collection(
    name="memora_memories"
)

In [58]:
save_memory(
    "memory_001",
    "My preferred programming language for AI projects is Python.",
    "preference",
    "user"
)

save_memory(
    "memory_002",
    "My internship application deadline is October 30.",
    "deadline",
    "internship_notice"
)

'Memory saved successfully.'

In [59]:
pages = extract_pdf_pages("test.pdf")

pdf_chunks = chunk_pdf_pages(pages)

print(add_pdf_chunks_to_memory(
    pdf_chunks,
    "test.pdf"
))

Added 52 chunks from test.pdf.


In [60]:
response = rag_answer(
    "what is environment?"
)

print("ANSWER:")
print(response["answer"])

print("\nSOURCES:")
print(response["sources"])

ANSWER:
Environment is defined as the sum total of water, air, land and the inter-relationships that exist among them and with the human beings, other living organisms and materials.

SOURCES:
[{'source': 'test.pdf', 'type': 'document_chunk', 'chunk_index': 0, 'page': 1}, {'page': 2, 'source': 'test.pdf', 'type': 'document_chunk', 'chunk_index': 6}, {'type': 'document_chunk', 'chunk_index': 2, 'source': 'test.pdf', 'page': 1}]


PHASE 4- QUERY

In [98]:
def classify_query(query):
    prompt = f"""
You are the query router for Memora, a personal knowledge and action assistant.

Classify the user's query into exactly ONE of these categories:

MEMORY
- Questions about personal information Memora has stored about the user.
- Example: "What is my favorite color?"
- Example: "What programming language do I prefer?"

RAG
- Questions that require information from uploaded documents or PDFs.
- Example: "What does the PDF say about the environment?"

ACTION
- Requests to CREATE or PERFORM an action.
- Example: "Remind me to apply for the internship."
- Example: "Create a task to finish my project."
- Example: "Remind me about my deadline tomorrow."

DATABASE
- Requests to READ, VIEW, CHECK, UPDATE, or DELETE existing stored tasks/actions.
- Example: "What tasks do I have?"
- Example: "Show my pending tasks."
- Example: "What reminders do I have?"
- Example: "Mark my internship task as completed."

Return ONLY the category name.

User query:
{query}
"""

    result = ask_llm(prompt)

    return result.strip().upper()

In [62]:
print(classify_query("What programming language do I prefer for AI projects?"))

RAG


In [63]:
print(classify_query("What does the PDF say about the environment?"))

RAG


In [64]:
print(classify_query("Remind me about my internship deadline tomorrow."))

ACTION


In [ ]:
def memora(query):
    route = classify_query(query)

    print("Route:", route)

    # ---------------- MEMORY ----------------
    if route == "MEMORY":
        results = search_memory(
            query,
            n_results=3,
            exclude_type="document_chunk"
        )

        return {
            "route": "MEMORY",
            "results": results
        }

    # ---------------- RAG ----------------
    elif route == "RAG":
        return {
            "route": "RAG",
            "response": rag_answer(query)
        }

    # ---------------- ACTION ----------------
    elif route == "ACTION":
        action_text = extract_action(query)
        action_data = parse_action(action_text)

        task_id = save_task(
            action_data["task"],
            action_data["date"],
            action_data["action_type"]
        )

        return {
            "route": "ACTION",
            "task_id": task_id,
            "action": action_data,
            "message": f"Task saved successfully with ID {task_id}."
        }

    # ---------------- DATABASE ----------------
    elif route == "DATABASE":
        db_action = classify_database_action(query)

        # VIEW TASKS
        if db_action == "VIEW":
            tasks = get_tasks()

            return {
                "route": "DATABASE",
                "action": "VIEW",
                "tasks": tasks
            }

        # COMPLETE TASK
        elif db_action == "COMPLETE":
            action_text = extract_action(query)
            action_data = parse_action(action_text)

            task_text = action_data["task"]

            result = complete_task_by_text(task_text)

            return {
                "route": "DATABASE",
                "action": "COMPLETE",
                "result": result
            }

        # DELETE TASK
        elif db_action == "DELETE":
            action_text = extract_action(query)
            action_data = parse_action(action_text)

            task_text = action_data["task"]

            task = find_task(task_text)

            if task is None:
                return {
                    "route": "DATABASE",
                    "action": "DELETE",
                    "message": "I couldn't find a matching pending task."
                }

            task_id = task[0]

            success = delete_task(task_id)

            return {
                "route": "DATABASE",
                "action": "DELETE",
                "task_id": task_id,
                "success": success,
                "message": (
                    f"Task '{task[1]}' deleted successfully."
                    if success
                    else "I couldn't delete the task."
                )
            }

        else:
            return {
                "route": "DATABASE",
                "action": "UNKNOWN",
                "message": f"Unknown database action: {db_action}"
            }

    return {
        "route": "UNKNOWN",
        "message": f"I couldn't classify the query: {query}"
    }

SyntaxError: unmatched '}' (3466368589.py, line 107)

In [66]:
response = memora(
    "What programming language do I prefer for AI projects?"
)

print(response)

Route: RAG
{'route': 'RAG', 'response': {'answer': "I couldn't find that information in your stored knowledge.", 'sources': [{'source': 'test.pdf', 'type': 'document_chunk', 'page': 9, 'chunk_index': 41}, {'page': 1, 'source': 'test.pdf', 'type': 'document_chunk', 'chunk_index': 1}, {'page': 8, 'chunk_index': 35, 'source': 'test.pdf', 'type': 'document_chunk'}]}}


In [67]:
response = memora(
    "What does the PDF say about the environment?"
)

print("Route:", response["route"])
print("Answer:", response["response"]["answer"])
print("Sources:", response["response"]["sources"])

Route: RAG
Route: RAG
Answer: The PDF states that the environment is the sum total of water, air, land, and the inter-relationships among them and with human beings, other living organisms, and materials. It also emphasizes that environment is a complex and multifaceted concept that deals with the working of the earth, its life support systems, interactions, influences, problems, and solutions.
Sources: [{'chunk_index': 0, 'source': 'test.pdf', 'page': 1, 'type': 'document_chunk'}, {'type': 'document_chunk', 'chunk_index': 6, 'page': 2, 'source': 'test.pdf'}, {'page': 1, 'type': 'document_chunk', 'source': 'test.pdf', 'chunk_index': 3}]


In [68]:
response = memora(
    "What programming language do I prefer for AI projects?"
)

print(response)

Route: RAG
{'route': 'RAG', 'response': {'answer': "I couldn't find that information in your stored knowledge.", 'sources': [{'type': 'document_chunk', 'page': 9, 'source': 'test.pdf', 'chunk_index': 41}, {'chunk_index': 1, 'source': 'test.pdf', 'type': 'document_chunk', 'page': 1}, {'page': 8, 'chunk_index': 35, 'type': 'document_chunk', 'source': 'test.pdf'}]}}


In [69]:
results = search_memory(
     "What programming language do I prefer for AI projects?",
    n_results=3,
    memory_type="document_chunk"
)

for i, chunk in enumerate(results["documents"][0]):
    print(f"\n--- RESULT {i+1} ---")
    print(chunk)
    print("METADATA:", results["metadatas"][0][i])


--- RESULT 1 ---
eper water of the ocean for 
operating OTE.  
 The warm surface water of ocean is used to b oil a liquid like ammonia.
METADATA: {'page': 9, 'chunk_index': 41, 'type': 'document_chunk', 'source': 'test.pdf'}

--- RESULT 2 ---
mation from various disciplines of science, social science, law and engineering have to be 
included in environmental studies to understand it completely. 
 Life sciences like botany, zoology, microbiology, genetics and bio technology etc help in 
understanding the biotic components and their interactions. 
 Physical sciences like physics, Chemistry, Earth Science, Oceanography etc help in 
understanding physical and chemical structure of abiotic components of environment along with 
mass and energy transfer. 
 Mathematics, statistics, computer science serve as effective tools in environmental modeling. 
 Subjects like economics, management and sociology provide the inputs for dealing with the socio 
economic aspects associated with various 

In [71]:
save_memory(
    "002",
    "My favorite color is pink.",
    "preference",
    "user"
)

'Memory saved successfully.'

In [73]:
results = search_memory(
    "What is my favorite color?",
    n_results=1,
    exclude_type="document_chunk"
)

print(results)

{'ids': [['002']], 'embeddings': None, 'documents': [['My favorite color is pink.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'type': 'preference', 'created_at': '2026-10-25T23:59:24.679524', 'source': 'user'}]], 'distances': [[0.47014668583869934]]}


In [74]:
response = memora("What is my favorite color?")
print(response)

Route: MEMORY
{'route': 'MEMORY', 'results': {'ids': [['002', 'memory_001', 'memory_002']], 'embeddings': None, 'documents': [['My favorite color is pink.', 'My preferred programming language for AI projects is Python.', 'My internship application deadline is October 30.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'created_at': '2026-10-25T23:59:24.679524', 'source': 'user', 'type': 'preference'}, {'source': 'user', 'created_at': '2026-10-25T23:54:47.252397', 'type': 'preference'}, {'created_at': '2026-10-25T23:54:47.381212', 'source': 'internship_notice', 'type': 'deadline'}]], 'distances': [[0.47014668583869934, 1.7255902290344238, 1.9624049663543701]]}}


memora_memories
├── personal memories
└── document chunks

memora_tasks
├── tasks
├── deadlines
└── reminders

In [78]:
def extract_action(query):
    prompt = f"""
You are the action extraction component of Memora.

Extract the actionable task from the user's request.

Return the result in exactly this format:

TASK: <task>
DATE: <date or NONE>
ACTION: <action type>

Examples:

User: Remind me to submit my internship application on October 30.
TASK: Submit my internship application
DATE: October 30
ACTION: reminder

User: Create a task to finish my AI project.
TASK: Finish my AI project
DATE: NONE
ACTION: task

User query:
{query}
"""

    return ask_llm(prompt)

In [79]:
print(extract_action(
    "Remind me to apply for the internship on October 30."
))

TASK: Apply for the internship
DATE: October 30
ACTION: reminder


In [82]:
def memora(query):
    return extract_action(query)

In [83]:
response = memora(
    "Remind me to apply for the internship on October 30."
)

print(response)

TASK: Apply for the internship
DATE: October 30
ACTION: reminder


TASK DATABASE

In [84]:
import sqlite3

conn = sqlite3.connect("memora_tasks.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    task TEXT NOT NULL,
    date TEXT,
    action_type TEXT,
    status TEXT DEFAULT 'pending',
    created_at TEXT
)
""")

conn.commit()

print("Task database ready!")

Task database ready!


In [85]:
from datetime import datetime

def save_task(task, date, action_type):
    created_at = datetime.now().isoformat()

    cursor.execute("""
        INSERT INTO tasks (task, date, action_type, status, created_at)
        VALUES (?, ?, ?, ?, ?)
    """, (
        task,
        date,
        action_type,
        "pending",
        created_at
    ))

    conn.commit()

    return cursor.lastrowid

In [86]:
task_id = save_task(
    "Apply for the internship",
    "October 30",
    "reminder"
)

print("Task ID:", task_id)

Task ID: 1


In [87]:
cursor.execute("SELECT * FROM tasks")

tasks = cursor.fetchall()

print(tasks)

[(1, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:16:22.833584')]


In [88]:
def parse_action(action_text):
    lines = action_text.strip().split("\n")

    action_data = {
        "task": None,
        "date": None,
        "action_type": None
    }

    for line in lines:
        if line.startswith("TASK:"):
            action_data["task"] = line.replace("TASK:", "").strip()

        elif line.startswith("DATE:"):
            action_data["date"] = line.replace("DATE:", "").strip()

        elif line.startswith("ACTION:"):
            action_data["action_type"] = line.replace("ACTION:", "").strip()

    if action_data["date"] == "NONE":
        action_data["date"] = None

    return action_data

In [89]:
action_text = extract_action(
    "Remind me to apply for the internship on October 30."
)

print(action_text)

TASK: Apply for the internship
DATE: October 30
ACTION: reminder


In [90]:
parsed = parse_action(action_text)

print(parsed)

{'task': 'Apply for the internship', 'date': 'October 30', 'action_type': 'reminder'}


In [92]:
response = memora(
    "Remind me to apply for the internship on October 30."
)

print(response)

Route: ACTION
{'route': 'ACTION', 'task_id': 2, 'action': {'task': 'Apply for the internship', 'date': 'October 30', 'action_type': 'reminder'}, 'message': 'Task saved successfully with ID 2.'}


In [93]:
cursor.execute("SELECT * FROM tasks")

for task in cursor.fetchall():
    print(task)

(1, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:16:22.833584')
(2, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:20:55.906162')


RETRIEVE THE TASKS 

In [94]:
def get_tasks(status="pending"):
    cursor.execute("""
        SELECT id, task, date, action_type, status, created_at
        FROM tasks
        WHERE status = ?
        ORDER BY id DESC
    """, (status,))

    return cursor.fetchall()

In [95]:
tasks = get_tasks()

for task in tasks:
    print(task)

(2, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:20:55.906162')
(1, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:16:22.833584')


In [97]:
response = memora("What tasks do I have?")

print(response)

Route: ACTION
{'route': 'ACTION', 'task_id': 3, 'action': {'task': 'Get tasks', 'date': None, 'action_type': 'query'}, 'message': 'Task saved successfully with ID 3.'}


In [102]:
def complete_task(task_id):
    cursor.execute("""
        UPDATE tasks
        SET status = 'completed'
        WHERE id = ?
    """, (task_id,))

    conn.commit()

    return cursor.rowcount > 0

In [103]:
print(complete_task(2))

True


In [104]:
cursor.execute("SELECT * FROM tasks")
tasks = cursor.fetchall()

for task in tasks:
    print(task)

(1, 'Apply for the internship', 'October 30', 'reminder', 'pending', '2026-10-26T00:16:22.833584')
(2, 'Apply for the internship', 'October 30', 'reminder', 'completed', '2026-10-26T00:20:55.906162')
(3, 'Get tasks', None, 'query', 'pending', '2026-10-26T00:24:44.896986')


In [105]:
def find_task(task_text):
    cursor.execute("""
        SELECT id, task, date, action_type, status
        FROM tasks
        WHERE task LIKE ?
        AND status = 'pending'
        ORDER BY id DESC
        LIMIT 1
    """, (f"%{task_text}%",))

    return cursor.fetchone()

In [106]:
task = find_task("internship")

print(task)

(1, 'Apply for the internship', 'October 30', 'reminder', 'pending')


In [107]:
def complete_task_by_text(task_text):
    task = find_task(task_text)

    if task is None:
        return {
            "success": False,
            "message": "I couldn't find a matching pending task."
        }

    task_id = task[0]

    success = complete_task(task_id)

    if success:
        return {
            "success": True,
            "task_id": task_id,
            "task": task[1],
            "message": f"Task '{task[1]}' marked as completed."
        }

    return {
        "success": False,
        "message": "I couldn't update the task."
    }

In [108]:
result = complete_task_by_text("internship")

print(result)

{'success': True, 'task_id': 1, 'task': 'Apply for the internship', 'message': "Task 'Apply for the internship' marked as completed."}


In [109]:
print(get_tasks())

[(3, 'Get tasks', None, 'query', 'pending', '2026-10-26T00:24:44.896986')]


In [110]:
def classify_database_action(query):
    prompt = f"""
You are the database action classifier for Memora.

Classify the user's request into exactly ONE of these:

VIEW
- User wants to see, list, or check existing tasks/reminders.
- Example: "What tasks do I have?"
- Example: "Show my pending reminders."

COMPLETE
- User wants to mark an existing task as completed.
- Example: "Mark my internship task as completed."
- Example: "I finished my internship application."

DELETE
- User wants to remove an existing task.
- Example: "Delete my old internship task."

Return ONLY one word:
VIEW, COMPLETE, or DELETE.

User query:
{query}
"""

    result = ask_llm(prompt)

    return result.strip().upper()

In [111]:
print(classify_database_action(
    "What tasks do I have?"
))

VIEW


In [112]:
print(classify_database_action(
    "Mark my internship task as completed."
))

COMPLETE


In [113]:
print(classify_database_action(
    "Delete my old internship task."
))

DELETE


In [115]:
print(memora("What is my favorite color?"))

Route: MEMORY


InternalError: Error executing plan: Internal error: Error finding id

In [ ]:
print(memora("What does the PDF say about the environment?"))

In [116]:
print("Collection:", memory_collection.name)
print("Count:", memory_collection.count())

Collection: memora_memories
Count: 55


In [117]:
print(memory_collection.peek())

InternalError: Error executing plan: Internal error: Error finding id

In [118]:
print("Collection name:", memory_collection.name)
print("Collection ID:", memory_collection.id)
print("Count:", memory_collection.count())

Collection name: memora_memories
Collection ID: 85f6f4f3-8fb3-4f65-b662-163909db13f9
Count: 55


In [121]:
print(chroma_client)
print(memory_collection)

Collection(name=memora_memories)


In [122]:
print("Current working directory:")
import os
print(os.getcwd())

Current working directory:
c:\Users\sharv\Desktop\Memora


In [123]:
import os

for root, dirs, files in os.walk(r"C:\Users\sharv\Desktop\Memora"):
    if "chroma.sqlite3" in files:
        print("FOUND:", os.path.join(root, "chroma.sqlite3"))

FOUND: C:\Users\sharv\Desktop\Memora\chroma_db\chroma.sqlite3


In [124]:
print("Old collection count:", memory_collection.count())

print("Collection metadata:")
print(memory_collection.metadata)

Old collection count: 55
Collection metadata:
None


In [125]:
print("Old collection ID:", memory_collection.id)

Old collection ID: 85f6f4f3-8fb3-4f65-b662-163909db13f9


In [126]:
data = memory_collection.get(limit=5)

print(data)

{'ids': ['memory_001', 'memory_002', 'test.pdf_0', 'test.pdf_1', 'test.pdf_2'], 'embeddings': None, 'documents': ['My preferred programming language for AI projects is Python.', 'My internship application deadline is October 30.', 'UNIT-1 \nINTRODUCTION TO ENVIRONMENT & NATURAL RESOURCES \nDepartment of Chemistry, ANITS  \nPage 1 \n \n \n\uf076 Environment: Environment is derived from the French word Environner, which means to \nencircle or surround. Environment is defined as the sum total of water, air, land and the inter-\nrelationships that exist among them and with the human beings, other living organisms and \nmaterials. \n\uf076 Multi-disciplinary nature of environmental Science \nEnvironment is complex and has multifarious aspects. \nEnvironmental sciences deal with working of earth, its life support systems, its interactions, \ninfluences, its problems, solutions. Keeping in view the complex nature of environment knowledge \nand information from various disciplines of science, 

In [128]:
data = memory_collection.get(
    limit=5,
    include=["documents", "metadatas", "embeddings"]
)

print("Documents:", len(data["documents"]))
print("Embeddings:", len(data["embeddings"]))
print("First embedding length:", len(data["embeddings"][0]))

InternalError: Error executing plan: Internal error: Error finding id

In [129]:
import shutil
import os

source = r"C:\Users\sharv\Desktop\Memora\chroma_db"
backup = r"C:\Users\sharv\Desktop\Memora\chroma_db_backup"

if not os.path.exists(backup):
    shutil.copytree(source, backup)
    print("Backup created successfully.")
else:
    print("Backup already exists.")

Backup created successfully.


In [131]:
data = memory_collection.get(
    include=["documents", "metadatas"]
)

print("Documents:", len(data["documents"]))
print("Metadatas:", len(data["metadatas"]))

Documents: 55
Metadatas: 55


In [132]:
new_collection = chroma_client.get_or_create_collection(
    name="memora_memories_v2"
)

print("New collection:", new_collection.name)

New collection: memora_memories_v2


In [133]:
new_embeddings = embedding_model.encode(
    data["documents"],
    show_progress_bar=True
).tolist()

print("Embeddings created:", len(new_embeddings))

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings created: 55


In [134]:
ids = data["ids"]

new_collection.add(
    ids=ids,
    documents=data["documents"],
    metadatas=data["metadatas"],
    embeddings=new_embeddings
)

print("All records restored!")

All records restored!


In [135]:
print("New collection count:", new_collection.count())

New collection count: 55


In [136]:
memory_collection = new_collection

print("Active collection:", memory_collection.name)
print("Records:", memory_collection.count())

Active collection: memora_memories_v2
Records: 55


In [137]:
results = search_memory(
    "What is my favorite color?",
    n_results=3,
    exclude_type="document_chunk"
)

print(results)

{'ids': [['002', 'memory_001', 'memory_002']], 'embeddings': None, 'documents': [['My favorite color is pink.', 'My preferred programming language for AI projects is Python.', 'My internship application deadline is October 30.']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'source': 'user', 'created_at': '2026-10-25T23:59:24.679524', 'type': 'preference'}, {'type': 'preference', 'created_at': '2026-10-25T23:54:47.252397', 'source': 'user'}, {'created_at': '2026-10-25T23:54:47.381212', 'source': 'internship_notice', 'type': 'deadline'}]], 'distances': [[0.4701468348503113, 1.7255901098251343, 1.9624052047729492]]}


In [139]:
response = memora("What is environment?")
print(response)

Route: RAG
{'route': 'RAG', 'response': {'answer': 'Environment is defined as the sum total of water, air, land and the inter-relationships that exist among them and with human beings, other living organisms and materials.', 'sources': [{'type': 'document_chunk', 'chunk_index': 0, 'page': 1, 'source': 'test.pdf'}, {'type': 'document_chunk', 'page': 2, 'chunk_index': 6, 'source': 'test.pdf'}, {'chunk_index': 2, 'page': 1, 'source': 'test.pdf', 'type': 'document_chunk'}]}}


In [140]:
def delete_task(task_id):
    cursor.execute("""
        DELETE FROM tasks
        WHERE id = ?
    """, (task_id,))

    conn.commit()

    return cursor.rowcount > 0

In [141]:
test_task_id = save_task(
    "Delete this test task",
    None,
    "task"
)

print("Test task ID:", test_task_id)

Test task ID: 4


In [142]:
print(delete_task(test_task_id))

True


In [143]:
cursor.execute(
    "SELECT * FROM tasks WHERE id = ?",
    (test_task_id,)
)

print(cursor.fetchone())

None
